# Classifier のハイパーパラメータ探索 (MNIST)

`compare_classifiers.ipynb` は既定値1点での比較だった。ここでは各 classifier を
**それぞれの best 設定まで持ち上げてから**比べる。

手順は2段階:

1. **units のグリッドサーチ** — 特徴量次元だけを 32〜1024 で振る (他は既定値)
2. **Optuna によるハイパーパラメータ最適化** — モデルごとの探索空間を TPE で探索。
   **積む層数 `n_layer` も探索対象に含める**

いずれも学習データから切り出した **validation で選び、test は最後に一度だけ見る**。
test で選ぶとリークするため。

リッジ回帰の正則化係数 `beta` は特別扱いする。正規方程式 `ZtZ`, `YtZ` を一度溜めれば
`beta` を変えて解き直すだけなので、**探索1試行あたり実質ゼロコストで全 beta を掃引できる**。
そのため beta は探索空間に入れず、常に内側で最良を選ぶ。

## 0. セットアップ

In [ ]:
import os
import sys
import time
import warnings

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append(os.path.abspath(".."))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import optuna
import torch
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

import networks
from src import classification as C

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# グラフの日本語ラベル用 (無い環境では豆腐になるので、その場合はこの行を消す)
plt.rcParams["font.family"] = "Noto Sans CJK JP"
optuna.logging.set_verbosity(optuna.logging.WARNING)

print("torch:", torch.__version__)
print("device:", DEVICE)
print("classifiers:", networks.list_classifiers())

## 1. 設定

In [ ]:
# ============================== ここを編集する ==============================
SEED = 0

DATA_ROOT = "~/torchvision_datasets"
N_TRAIN = 60_000  # 学習データ全体 (validation はここから切り出す)
N_TEST = 10_000
VAL_RATIO = 0.2  # validation に回す割合

BATCH_SIZE = 256

# beta は探索空間に入れず、各試行の内側で常にこの一覧から最良を選ぶ (追加コストはほぼゼロ)
BETAS = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0]

MODELS = ["esn", "bi_esn", "bi_esn2d", "conv2d", "reservoir_conv2d"]

# --- 1段目: units のグリッドサーチ ---
FEATURE_DIMS = [32, 64, 128, 256, 512, 1024]

# --- 2段目: Optuna ---
N_TRIALS = 40
TIME_BUDGET = 600  # モデルあたりの探索時間の上限 [秒]。深い構成は 1 試行が重いので必ず入れる
SEARCH_N_TRAIN = 15_000  # 探索は学習枚数を絞って高速化し、最良設定だけ全データで再学習する
MAX_LAYERS = 3  # 探索する層数の上限
MIN_BATCH = 8  # OOM 時にバッチを半分ずつ下げる下限
# ===========================================================================

## 2. データ

学習データを train / validation に層化分割する。探索中は test を一切見ない。

In [ ]:
x_all, y_all, x_test, y_test, NUM_CLASSES = C.load_dataset("mnist", DATA_ROOT)
x_all, y_all = x_all[:N_TRAIN], y_all[:N_TRAIN]
x_test, y_test = x_test[:N_TEST], y_test[:N_TEST]

tr_idx, va_idx = train_test_split(
    np.arange(len(y_all)), test_size=VAL_RATIO, random_state=SEED, stratify=y_all
)
x_tr, y_tr = x_all[tr_idx], y_all[tr_idx]
x_va, y_va = x_all[va_idx], y_all[va_idx]

y_tr_oh = F.one_hot(torch.from_numpy(y_tr), NUM_CLASSES).float()
y_all_oh = F.one_hot(torch.from_numpy(y_all), NUM_CLASSES).float()

INPUT_SHAPE = tuple(x_all.shape[1:])
print(f"train {len(x_tr)} / val {len(x_va)} / test {len(x_test)}   input_shape={INPUT_SHAPE}")

## 3. 共通処理

`beta` の掃引を内側に閉じ込めた学習・評価。

In [ ]:
@torch.no_grad()
def fit_select_beta(model, x_fit, y_fit_oh, x_sel, y_sel, batch_size=BATCH_SIZE, betas=BETAS):
    """x_fit で正規方程式を溜め、x_sel で beta を選んで読み出しにセットする。

    Returns: (best_beta, 選択に使った精度)
    """
    D, K = int(model.feature_dim), int(y_fit_oh.shape[1])
    ZTZ = torch.zeros(D, D, dtype=torch.float64, device=DEVICE)
    YTZ = torch.zeros(K, D, dtype=torch.float64, device=DEVICE)

    start = 0
    for xb in C.iter_batches(x_fit, batch_size, DEVICE):
        yb = y_fit_oh[start : start + len(xb)].to(DEVICE).double()
        zb = model.features(xb).double()
        ZTZ += zb.T @ zb
        YTZ += yb.T @ zb
        start += len(xb)

    # 選択用の特徴量は使い回すので一度だけ計算する
    Z_sel = torch.cat([model.features(b).double() for b in C.iter_batches(x_sel, batch_size, DEVICE)])
    eye = torch.eye(D, dtype=torch.float64, device=DEVICE)

    best = (None, -1.0, None)
    for beta in betas:
        W = torch.linalg.solve(ZTZ + beta * eye, YTZ.T)
        acc = float(((Z_sel @ W).argmax(1).cpu().numpy() == y_sel).mean())
        if acc > best[1]:
            best = (beta, acc, W)

    model.readout.set_kernel(best[2].float())

    return best[0], best[1]


def build(name, kwargs, seed=None):
    C.set_global_determinism(SEED if seed is None else seed)
    model = networks.build_classifier(
        name, input_shape=INPUT_SHAPE, num_classes=NUM_CLASSES, seed=SEED if seed is None else seed, **kwargs
    )
    return model.to(DEVICE).eval()


def evaluate_config(name, kwargs, n_fit=None, report_test=True):
    """train で学習し val で beta を選ぶ。report_test なら test も評価する。

    NOTE: 層を積むと中間テンソルが一気に大きくなる (特に reservoir_conv2d) ため、
          OOM したらバッチを半分にして再試行する。
    """
    n_fit = len(x_tr) if n_fit is None else n_fit
    batch_size = BATCH_SIZE

    while batch_size >= MIN_BATCH:
        model = None
        try:
            t0 = time.time()
            torch.cuda.reset_peak_memory_stats()

            model = build(name, kwargs)
            beta, val_acc = fit_select_beta(model, x_tr[:n_fit], y_tr_oh[:n_fit], x_va, y_va, batch_size)

            out = {
                "model": name,
                "feature_dim": int(model.feature_dim),
                "n_layer": int(kwargs.get("n_layer", 1)),
                "beta": beta,
                "val_acc": val_acc,
                "batch_size": batch_size,
                "peak_mib": torch.cuda.max_memory_allocated() / 2**20,
                "elapsed": time.time() - t0,
            }
            if report_test:
                out.update(C.evaluate_model(model, x_test, y_test, NUM_CLASSES, batch_size, DEVICE))

            del model
            torch.cuda.empty_cache()

            return out
        except torch.OutOfMemoryError:
            del model
            torch.cuda.empty_cache()
            batch_size //= 2

    raise torch.OutOfMemoryError(f"{name} {kwargs} はバッチ {MIN_BATCH} でも載らない")


def size_kwargs(name, dim):
    """目標の特徴量次元に合わせたサイズ引数 (他は既定値)。"""
    if name in ("esn", "bi_esn", "bi_esn2d"):
        return {"units": dim, "patch_sizes": (4, 4)}
    if name == "conv2d":
        return {"filters": dim, "kernel_size": 3, "activations": "tanh"}

    # reservoir_conv2d の特徴量次元は 2 * num_reservoirs * units
    return {"num_reservoirs": 5, "units": max(1, round(dim / 10)), "kernel_size": 3}

## 4. units のグリッドサーチ

特徴量次元だけを振る。他のハイパーパラメータは既定値のまま。

In [ ]:
grid = {name: [] for name in MODELS}
for dim in FEATURE_DIMS:
    line = f"dim={dim:5d} |"
    for name in MODELS:
        r = evaluate_config(name, size_kwargs(name, dim))
        grid[name].append(r)
        line += f" {name}={r['acc']:.4f}"
    print(line, flush=True)

plt.figure(figsize=(6.5, 4))
for name, rows in grid.items():
    plt.plot([r["feature_dim"] for r in rows], [r["acc"] for r in rows], "o-", label=name)
plt.axhline(1 / NUM_CLASSES, color="crimson", ls="--", lw=1)
plt.xscale("log", base=2)
plt.xlabel("特徴量の次元")
plt.ylabel("test accuracy")
plt.title("units グリッドサーチ (他は既定値)")
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Optuna によるハイパーパラメータ最適化

モデルごとに探索空間を定義する。選択基準は validation accuracy。

In [ ]:
DIMS = FEATURE_DIMS


def suggest(trial, name):
    """モデルごとの探索空間。beta は含めない (内側で常に最良を選ぶため)。"""
    n_layer = trial.suggest_int("n_layer", 1, MAX_LAYERS)

    if name in ("esn", "bi_esn", "bi_esn2d"):
        patch = trial.suggest_categorical("patch", [(2, 2), (4, 4), (7, 7)])
        return {
            "n_layer": n_layer,
            "units": trial.suggest_categorical("units", DIMS),
            "patch_sizes": patch,
            "connectivity": trial.suggest_float("connectivity", 0.02, 0.9, log=True),
            "leaky": trial.suggest_float("leaky", 0.3, 1.0),
            "spectral_radius": trial.suggest_float("spectral_radius", 0.3, 1.3),
        }
    if name == "conv2d":
        return {
            "n_layer": n_layer,
            "filters": trial.suggest_categorical("filters", DIMS),
            "kernel_size": trial.suggest_categorical("kernel_size", [3, 5, 7]),
            "activations": trial.suggest_categorical("activations", ["tanh", "relu", "gelu", "sigmoid"]),
        }

    return {
        "n_layer": n_layer,
        "num_reservoirs": trial.suggest_categorical("num_reservoirs", [3, 5, 8]),
        "units": trial.suggest_categorical("units", [4, 8, 12, 16, 24, 32]),
        "kernel_size": trial.suggest_categorical("kernel_size", [3, 5]),
        "connectivity": trial.suggest_float("connectivity", 0.02, 0.9, log=True),
        "spectral_radius": trial.suggest_float("spectral_radius", 0.3, 1.3),
    }


studies = {}
for name in MODELS:
    t0 = time.time()

    def objective(trial, name=name):
        try:
            return evaluate_config(name, suggest(trial, name), n_fit=SEARCH_N_TRAIN, report_test=False)["val_acc"]
        except torch.OutOfMemoryError:
            # メモリに載らない構成は失敗ではなく「探索対象外」として打ち切る
            raise optuna.TrialPruned() from None

    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(objective, n_trials=N_TRIALS, timeout=TIME_BUDGET)
    studies[name] = study

    n_done = len([t for t in study.trials if t.value is not None])
    print(
        f"{name:<17s} best val_acc={study.best_value:.4f}  {n_done}/{len(study.trials)} 試行完了  "
        f"{time.time() - t0:5.1f}s\n{'':19s}params={study.best_params}",
        flush=True,
    )

## 6. 最良設定を全データで再学習して比較

探索は `SEARCH_N_TRAIN` 枚で行ったので、選ばれた設定を学習データ全体で学習し直してから test を見る。

In [ ]:
def best_kwargs(name, params):
    """Optuna の best_params を build_classifier の引数に戻す。"""
    params = dict(params)
    if name in ("esn", "bi_esn", "bi_esn2d"):
        params["patch_sizes"] = tuple(params.pop("patch"))
    return params


final = []
for name in MODELS:
    kwargs = best_kwargs(name, studies[name].best_params)

    t0 = time.time()
    model = build(name, kwargs)
    beta, val_acc = fit_select_beta(model, x_tr, y_tr_oh, x_va, y_va)
    metrics = C.evaluate_model(model, x_test, y_test, NUM_CLASSES, BATCH_SIZE, DEVICE)

    grid_best = max(grid[name], key=lambda r: r["acc"])
    final.append(
        {
            "model": name,
            "feature_dim": int(model.feature_dim),
            "beta": beta,
            "val_acc": val_acc,
            "grid_acc": grid_best["acc"],
            "elapsed": time.time() - t0,
            **metrics,
            "params": kwargs,
        }
    )
    del model
    torch.cuda.empty_cache()

    print(f"{name:<17s} test acc={metrics['acc']:.4f} (grid best {grid_best['acc']:.4f})", flush=True)

In [ ]:
header = (
    f"{'model':<18s}{'layers':>8s}{'dim':>6s}{'beta':>8s}"
    f"{'val':>8s}{'test':>8s}{'macro_f1':>10s}{'grid best':>11s}"
)
print(header)
print("-" * len(header))
for r in sorted(final, key=lambda r: -r["acc"]):
    print(
        f"{r['model']:<18s}{r['params'].get('n_layer', 1):>8d}{r['feature_dim']:>6d}{r['beta']:>8.0e}"
        f"{r['val_acc']:>8.4f}{r['acc']:>8.4f}{r['macro_f1']:>10.4f}{r['grid_acc']:>11.4f}"
    )

print("\n最良設定:")
for r in sorted(final, key=lambda r: -r["acc"]):
    print(f"  {r['model']:<17s} {r['params']}")

names = [r["model"] for r in final]
pos = np.arange(len(names))
plt.figure(figsize=(8.5, 3.8))
plt.bar(pos - 0.2, [r["grid_acc"] for r in final], 0.4, label="units グリッドの最良")
plt.bar(pos + 0.2, [r["acc"] for r in final], 0.4, label="Optuna 最適化後")
plt.axhline(1 / NUM_CLASSES, color="crimson", ls="--", lw=1, label=f"偶然一致 ({1 / NUM_CLASSES:.1f})")
plt.xticks(pos, names, rotation=15)
plt.ylabel("test accuracy")
plt.ylim(0, 1)
plt.title(f"MNIST (train={len(x_tr)}, test={N_TEST}, {N_TRIALS} trials)")
plt.legend(fontsize=9)
plt.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 7. 層数の効果

探索が選んだ層数は「その構成での最良の1点」でしかない。層数だけを振ったときの曲線を見ると、
モデルごとに層を重ねる意味が違うことが分かる。

他のハイパーパラメータは各モデルの best 設定に固定し、`n_layer` だけを変える。
`beta` は層数ごとに選び直す (最適な正則化は層数で変わるため)。

In [ ]:
depth = {name: [] for name in MODELS}
for name in MODELS:
    kwargs = best_kwargs(name, studies[name].best_params)
    for n_layer in range(1, MAX_LAYERS + 1):
        try:
            r = evaluate_config(name, dict(kwargs, n_layer=n_layer))
        except torch.OutOfMemoryError:
            print(f"  {name:<17s} n_layer={n_layer}: メモリ不足のため測定できず", flush=True)
            continue
        depth[name].append(r)
        print(
            f"  {name:<17s} n_layer={n_layer} test={r['acc']:.4f} "
            f"({r['elapsed']:5.1f}s, peak {r['peak_mib']:6.0f} MiB, batch {r['batch_size']})",
            flush=True,
        )

plt.figure(figsize=(6, 4))
for name, rows in depth.items():
    if rows:
        plt.plot([r["n_layer"] for r in rows], [r["acc"] for r in rows], "o-", label=name)
plt.xlabel("層数 (n_layer)")
plt.ylabel("test accuracy")
plt.xticks(range(1, MAX_LAYERS + 1))
plt.title("層数だけを振った場合 (他は best 設定)")
plt.legend(fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. 探索の中身を見る

どのハイパーパラメータが効いたのかを確認する。

In [ ]:
for name in MODELS:
    study = studies[name]
    try:
        importance = optuna.importance.get_param_importances(study)
    except Exception as e:  # 試行数が少ない・全試行が同値などで計算できない場合
        print(f"{name}: 重要度を計算できず ({type(e).__name__})")
        continue

    top = "  ".join(f"{k}={v:.2f}" for k, v in list(importance.items())[:4])
    values = [t.value for t in study.trials if t.value is not None]
    print(f"{name:<17s} val_acc {min(values):.4f}-{max(values):.4f}  重要度: {top}")